# 4대 RAG 평가자

Python 3.12 새 커널에서 네 평가자의 비교 대상과 정답 답변 필요 여부부터 확인합니다.

## Python 3.12 패키지 버전 고정

In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.

## 네 평가자의 비교 대상과 정답 답변 필요 여부

In [2]:
# 평가자마다 비교하는 두 대상과 정답 답변 필요 여부를 표로 정리합니다.
evaluators = {
 'Correctness': ('응답', '정답 답변', True),
 'Relevance': ('응답', '질문', False),
 'Groundedness': ('응답', '검색 문서', False),
 'Retrieval relevance': ('검색 문서', '질문', False),
}
for name, (left, right, reference) in evaluators.items():
    print(f'{name:20} {left} vs {right} | reference={reference}')

Correctness          응답 vs 정답 답변 | reference=True
Relevance            응답 vs 질문 | reference=False
Groundedness         응답 vs 검색 문서 | reference=False
Retrieval relevance  검색 문서 vs 질문 | reference=False


## Q002 질문·응답·문서 입력 확인

In [3]:
# Q002의 질문·응답 단어와 출처 ID를 사용해 코드 평가 구조를 확인합니다.
from pathlib import Path
import json, re
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'labs/part4/ch15-evaluation-advanced').is_dir())
data = json.loads((root / 'labs/part4/ch15-evaluation-advanced/data/evaluation-cases.json').read_text())
case = next(x for x in data['cases'] if x['qid'] == 'Q002')
def words(text: str) -> set[str]:
    return set(re.findall(r'[가-힣A-Za-z0-9]+', text))
print('질문-응답 겹침:', round(len(words(case['question']) & words(case['response'])) / len(words(case['question'])), 3))
print('출처 표기:', all(doc in case['response'] for doc in case['retrieved_doc_ids']))

질문-응답 겹침: 0.5
출처 표기: True


## 교육용 근사값과 실제 근거 충실도 평가 구분

In [4]:
# 실제 judge 출력과 고정한 실행 모델·프롬프트 버전을 읽습니다.
verdict = json.loads(case['judge_raw'])
correctness = len(words(case['reference_answer']) & words(case['response'])) / len(words(case['reference_answer']))
relevance = len(words(case['question']) & words(case['response'])) / len(words(case['question']))
retrieval = len(words(case['question']) & words(case['context'])) / len(words(case['question']))
scores = {'Correctness 교육용 근사': round(correctness, 3),
          'Relevance 교육용 근사': round(relevance, 3),
          'Groundedness 실제 judge': verdict['score'],
          'Retrieval relevance 교육용 근사': round(retrieval, 3)}
for name, score in scores.items():
    print(name, score)
print(case['reference_kind'], case['judge_model'], case['judge_prompt_version'])
assert case['judge_prompt_version'] == 'groundedness-v1.0.0'

Correctness 교육용 근사 0.667
Relevance 교육용 근사 0.5
Groundedness 실제 judge 1
Retrieval relevance 교육용 근사 0.5
교육용 작성자 기준 gpt-5.4-mini groundedness-v1.0.0
